# Phase 3 — Speaker Embeddings (educational, UNTRAINED encoder)

```
WAV -> 16 kHz mono -> Mel spectrogram -> SpeakerEncoder (PyTorch) -> 128-D embedding -> cosine similarity
```

> **Important:** the encoder in this notebook has **random weights**. It always produces a 128-number
> vector, but that vector is **not** a meaningful speaker embedding until the network is trained on many speakers.
> Nothing here recognizes your voice.

## 1. Device: GPU if available, otherwise CPU (both work for this small experiment)

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. Clone the repository and install requirements

In [ ]:
import os
if not os.path.exists("/content/Voice_engine"):
    !git clone https://github.com/8919134556/Voice_engine.git /content/Voice_engine
%cd /content/Voice_engine
!git pull
!pip install -q -r requirements.txt

## 3. Private dataset

**Option A — Google Drive (recommended):** `MyDrive/voice_engine/dataset/speaker_01/audio_00X.wav`

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DATASET_DIR = "/content/drive/MyDrive/voice_engine/dataset"
!ls "{DATASET_DIR}/speaker_01"

**Option B — upload for this session only** (skip if you used Drive).

In [ ]:
# Run ONLY if you are not using Google Drive.
import os, shutil
from google.colab import files
DATASET_DIR = "/content/Voice_engine/dataset"
os.makedirs(f"{DATASET_DIR}/speaker_01", exist_ok=True)
for name in files.upload():
    shutil.move(name, f"{DATASET_DIR}/speaker_01/{name}")
!ls "{DATASET_DIR}/speaker_01"

## 4. Run the whole Phase 3 experiment
Embeddings are written to `/content/Voice_engine/embeddings/speaker_01/*.npy` (git-ignored).

In [ ]:
!python scripts/generate_speaker_embeddings.py --dataset "{DATASET_DIR}"

In [ ]:
from IPython.display import Image
Image("outputs/phase3/embeddings_pca.png")

---
# Step by step

### A. WAV → Mel spectrogram → encoder input `[batch, channel, mel bands, time]`

In [ ]:
from pathlib import Path
import numpy as np
from src.audio.loader import list_audio_files
from src.embeddings.generate_embeddings import wav_to_encoder_input, embed_mel, waveform_to_encoder_input
from src.models.speaker_encoder import SpeakerEncoder, count_parameters
from src.similarity.cosine import cosine_similarity, pairwise_similarities

wav_files = list_audio_files(Path(DATASET_DIR) / "speaker_01")
for p in wav_files:
    print(p.name, "->", list(wav_to_encoder_input(p).shape))
print("\nThe last number (time frames) differs because the recordings have different lengths (100 frames = 1 second).")

### B. The encoder, layer by layer — watch the shape change

In [ ]:
torch.manual_seed(0)
model = SpeakerEncoder(embedding_dim=128).to(device).eval()
print(model)
print(f"\nTrainable parameters: {count_parameters(model):,}")

x = wav_to_encoder_input(wav_files[0]).to(device)
print("\ninput           ", list(x.shape))
with torch.no_grad():
    x = model.relu(model.conv1(x)); print("conv1 + ReLU    ", list(x.shape))
    x = model.relu(model.conv2(x)); print("conv2 + ReLU    ", list(x.shape))
    x = model.pool(x);              print("adaptive pool   ", list(x.shape), " <- time is gone: any length works")
    x = model.flatten(x);           print("flatten         ", list(x.shape))
    x = model.fc(x);                print("linear          ", list(x.shape), " <- the 128-D embedding")

### C. Embeddings + cosine similarity (untrained!)

In [ ]:
embeddings = {p.stem: embed_mel(model, wav_to_encoder_input(p), device) for p in wav_files}
for name, e in embeddings.items():
    print(f"{name}: shape {e.shape}, length {np.linalg.norm(e):.3f}, first values {np.round(e[:5], 3)}")
print()
for a, b, sim in pairwise_similarities(embeddings):
    print(f"{a} vs {b}: {sim:.4f}")

### D. Proof that the untrained vectors are NOT speaker embeddings

1. Non-voice sounds score almost as "similar" as your own recordings.
2. A different random seed gives a completely different vector for the **same** file.

In [ ]:
sr = 16000
t = torch.arange(3 * sr) / sr
noise = 0.1 * torch.randn(1, len(t))
beep = 0.3 * torch.sin(2 * torch.pi * 440 * t).unsqueeze(0)
first = next(iter(embeddings))
for name, w in [("white noise", noise), ("440 Hz beep", beep)]:
    e = embed_mel(model, waveform_to_encoder_input(w, sr), device)
    print(f"{first} vs {name}: {cosine_similarity(embeddings[first], e):.4f}")

torch.manual_seed(1)
other_model = SpeakerEncoder().to(device).eval()
e_other = embed_mel(other_model, wav_to_encoder_input(wav_files[0]), device)
print(f"\nSame file, seed 0 vs seed 1 network: {cosine_similarity(embeddings[first], e_other):.4f}")
print("-> the numbers depend on the random weights, not on who is speaking.")

### E. Concept preview — a speaker profile (NOT the Phase 5 Voice Registry)
Average the recording embeddings → one representative vector for the speaker.
Only meaningful once the encoder is trained; shown here just to illustrate the idea.

In [ ]:
profile = np.mean(list(embeddings.values()), axis=0)
profile = profile / np.linalg.norm(profile)
print("profile shape:", profile.shape)
for name, e in embeddings.items():
    print(f"{name} vs speaker_01 profile: {cosine_similarity(e, profile):.4f}")

### Privacy
The `.npy` embeddings are **derived biometric data**. They are git-ignored. To keep them between Colab sessions,
copy them to your private Drive — never to GitHub:

`!cp -r embeddings /content/drive/MyDrive/voice_engine/`